In [1]:
import pandas as pd

# ==========================================
# قراءة البيانات
# ==========================================
df = pd.read_csv("Saudi_Ecommerce.csv", sep=";")


# ==========================================
# 1. فحص القيم الفارغة والمخفية
# ==========================================
print("=" * 60)
print("BLANK & EMPTY VALUES CHECK")
print("=" * 60)

for column in df.columns:
    blank_count = (
        df[column]
        .astype("string")
        .str.strip()
        .eq("")
        .sum()
    )

    print(f"{column}: {blank_count} blank values")


# ==========================================
# 2. فحص القيم Null + Blank معًا
# ==========================================
print("\n" + "=" * 60)
print("NULL + BLANK VALUES")
print("=" * 60)

for column in df.columns:
    null_count = df[column].isna().sum()

    blank_count = (
        df[column]
        .astype("string")
        .str.strip()
        .eq("")
        .sum()
    )

    total_missing = null_count + blank_count

    print(
        f"{column}: "
        f"Null={null_count}, "
        f"Blank={blank_count}, "
        f"Total Missing={total_missing}"
    )


# ==========================================
# 3. فحص التكرارات الكاملة
# ==========================================
print("\n" + "=" * 60)
print("FULL DUPLICATE RECORDS")
print("=" * 60)

duplicates = df[df.duplicated(keep=False)]

print(f"Total duplicated rows: {len(duplicates)}")

if len(duplicates) > 0:
    print("\nDuplicated records:")
    print(duplicates)


# ==========================================
# 4. فحص أسماء المتاجر المكررة
# ==========================================
print("\n" + "=" * 60)
print("DUPLICATE STORE NAMES")
print("=" * 60)

name_duplicates = (
    df[df["Name_AR"].duplicated(keep=False)]
    .sort_values("Name_AR")
)

print(f"Duplicated Arabic names: {len(name_duplicates)}")

print(name_duplicates[
    ["Name_AR", "Name_ENG", "CR", "Activity"]
].head(30))


# ==========================================
# 5. توزيع Activity
# ==========================================
print("\n" + "=" * 60)
print("ACTIVITY DISTRIBUTION")
print("=" * 60)

activity_count = df["Activity"].value_counts()
activity_percent = df["Activity"].value_counts(normalize=True) * 100

activity_report = pd.DataFrame({
    "Count": activity_count,
    "Percentage": activity_percent.round(2)
})

print(activity_report)


# ==========================================
# 6. توزيع Category
# ==========================================
print("\n" + "=" * 60)
print("CATEGORY DISTRIBUTION")
print("=" * 60)

category_count = df["Category"].value_counts()
category_percent = df["Category"].value_counts(normalize=True) * 100

category_report = pd.DataFrame({
    "Count": category_count,
    "Percentage": category_percent.round(2)
})

print(category_report)


# ==========================================
# 7. فحص Rating
# ==========================================
print("\n" + "=" * 60)
print("RATING CHECK")
print("=" * 60)

print("Unique Rating values:")
print(sorted(df["Rating"].dropna().unique()))

print("\nRating frequency:")
print(df["Rating"].value_counts().sort_index())


# ==========================================
# 8. فحص Rating خارج النطاق المتوقع
# ==========================================
print("\n" + "=" * 60)
print("INVALID RATINGS")
print("=" * 60)

invalid_rating = df[
    (df["Rating"] < 0) |
    (df["Rating"] > 10)
]

print(f"Invalid Rating values: {len(invalid_rating)}")

if len(invalid_rating) > 0:
    print(invalid_rating[["Name_AR", "Rating"]])


# ==========================================
# 9. فحص Num_Ratings
# ==========================================
print("\n" + "=" * 60)
print("NUM_RATINGS CHECK")
print("=" * 60)

print(df["Num_Ratings"].describe())

print("\nNum_Ratings frequency:")
print(df["Num_Ratings"].value_counts().sort_index())


# ==========================================
# 10. فحص Num_Ratings السالبة
# ==========================================
print("\n" + "=" * 60)
print("NEGATIVE NUM_RATINGS")
print("=" * 60)

negative_reviews = df[
    df["Num_Ratings"] < 0
]

print(f"Negative values: {len(negative_reviews)}")


# ==========================================
# 11. العلاقة المنطقية بين Rating و Num_Ratings
# ==========================================
print("\n" + "=" * 60)
print("RATING vs NUM_RATINGS LOGIC CHECK")
print("=" * 60)

# متاجر لديها تقييم ولكن لا يوجد لديها أي تقييمات
case_1 = df[
    (df["Rating"] > 0) &
    (df["Num_Ratings"] == 0)
]

print(
    "Rating > 0 but Num_Ratings = 0:",
    len(case_1)
)

# متاجر لديها تقييمات ولكن Rating = 0
case_2 = df[
    (df["Rating"] == 0) &
    (df["Num_Ratings"] > 0)
]

print(
    "Rating = 0 but Num_Ratings > 0:",
    len(case_2)
)


# ==========================================
# 12. فحص CR
# ==========================================
print("\n" + "=" * 60)
print("CR CHECK")
print("=" * 60)

print("CR data type:", df["CR"].dtype)
print("CR available:", df["CR"].notna().sum())
print("CR missing:", df["CR"].isna().sum())

print("\nCR unique values:")
print(df["CR"].dropna().nunique())


# ==========================================
# 13. فحص أرقام الجوال
# ==========================================
print("\n" + "=" * 60)
print("PHONE NUMBER CHECK")
print("=" * 60)

phone_as_string = df["Phone Number"].astype("string")

print("Phone length distribution:")
print(
    phone_as_string
    .str.len()
    .value_counts()
    .sort_index()
)

invalid_phone = df[
    phone_as_string.str.len() != 9
]

print(
    "\nPhone numbers not containing 9 digits:",
    len(invalid_phone)
)


# ==========================================
# 14. فحص البريد الإلكتروني
# ==========================================
print("\n" + "=" * 60)
print("EMAIL CHECK")
print("=" * 60)

email_as_string = df["Email"].astype("string").str.strip()

invalid_email = df[
    ~email_as_string.str.contains("@", na=False)
]

print(
    "Emails without @:",
    len(invalid_email)
)


# ==========================================
# 15. فحص الروابط
# ==========================================
print("\n" + "=" * 60)
print("URL CHECK")
print("=" * 60)

for column in ["Website", "Instagram", "Twitter"]:

    values = df[column].astype("string")

    invalid_urls = values[
        ~values.str.contains(
            "http|www|\\.com|\\.sa|\\.net|\\.org|t\\.me",
            case=False,
            na=False,
            regex=True
        )
        &
        values.notna()
        &
        values.str.strip().ne("")
    ]

    print(
        f"{column}: {len(invalid_urls)} suspicious values"
    )


# ==========================================
# 16. ملخص الفحص المتقدم
# ==========================================
print("\n" + "=" * 60)
print("ADVANCED INSPECTION COMPLETED")
print("=" * 60)

print("No data has been modified.")
print("No rows have been deleted.")
print("No values have been replaced.")

BLANK & EMPTY VALUES CHECK
Name_ENG: 0 blank values
Name_AR: 0 blank values
CR: 0 blank values
Category: 0 blank values
Website: 0 blank values
Instagram: 0 blank values
Twitter: 0 blank values
Email: 0 blank values
Phone Number: 0 blank values
Activity: 0 blank values
Rating: 0 blank values
Num_Ratings: 0 blank values
Unnamed: 12: 0 blank values

NULL + BLANK VALUES
Name_ENG: Null=391, Blank=0, Total Missing=391
Name_AR: Null=0, Blank=0, Total Missing=0
CR: Null=3491, Blank=0, Total Missing=3491
Category: Null=0, Blank=0, Total Missing=0
Website: Null=0, Blank=0, Total Missing=0
Instagram: Null=3788, Blank=0, Total Missing=3788
Twitter: Null=4662, Blank=0, Total Missing=4662
Email: Null=0, Blank=0, Total Missing=0
Phone Number: Null=0, Blank=0, Total Missing=0
Activity: Null=0, Blank=0, Total Missing=0
Rating: Null=0, Blank=0, Total Missing=0
Num_Ratings: Null=0, Blank=0, Total Missing=0
Unnamed: 12: Null=5401, Blank=0, Total Missing=5401

FULL DUPLICATE RECORDS
Total duplicated rows: